# DBSCAN — Density-Based Spatial Clustering of Applications with Noise

DBSCAN (Ester et al., 1996) defines clusters as **dense regions** separated by sparse regions. Two parameters:

* $\varepsilon$ (`eps`): neighbourhood radius
* `min_samples`: number of points (including the point itself) needed within $\varepsilon$ for a region to be dense

Point types:
* **Core point**: has at least `min_samples` points in its $\varepsilon$-neighbourhood.
* **Border point**: not core, but within $\varepsilon$ of a core point.
* **Noise point**: neither — labelled $-1$.

**Algorithm**: for each unvisited core point, start a new cluster and expand it with a breadth-first search through
all **density-reachable** points (neighbours of core points join the cluster; only core points expand further).

Advantages: no need to choose $k$, finds arbitrarily shaped clusters, robust to outliers.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

from collections import deque

In [ ]:
def make_blobs(n_samples=300, centers=3, cluster_std=1.0, n_features=2,
               center_box=(-10, 10), random_state=None):
    """Isotropic Gaussian blobs (re-implementation of sklearn's make_blobs)."""
    rng = np.random.RandomState(random_state)
    if isinstance(centers, int):
        centers = rng.uniform(*center_box, size=(centers, n_features))
    centers = np.asarray(centers, dtype=float)
    k = len(centers)
    counts = [n_samples // k + (1 if i < n_samples % k else 0) for i in range(k)]
    stds = np.broadcast_to(np.asarray(cluster_std, dtype=float), (k,))
    X = np.vstack([rng.normal(c, s, size=(m, centers.shape[1]))
                   for c, s, m in zip(centers, stds, counts)])
    y = np.concatenate([np.full(m, i) for i, m in enumerate(counts)])
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def make_moons(n_samples=200, noise=0.1, random_state=None):
    """Two interleaving half circles."""
    rng = np.random.RandomState(random_state)
    n_out = n_samples // 2
    n_in = n_samples - n_out
    t_out = np.linspace(0, np.pi, n_out)
    t_in = np.linspace(0, np.pi, n_in)
    outer = np.c_[np.cos(t_out), np.sin(t_out)]
    inner = np.c_[1 - np.cos(t_in), 0.5 - np.sin(t_in)]
    X = np.vstack([outer, inner]) + rng.normal(0, noise, size=(n_samples, 2))
    y = np.r_[np.zeros(n_out, dtype=int), np.ones(n_in, dtype=int)]
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def make_circles(n_samples=200, factor=0.5, noise=0.05, random_state=None):
    """A large circle containing a smaller circle."""
    rng = np.random.RandomState(random_state)
    n_out = n_samples // 2
    n_in = n_samples - n_out
    t_out = rng.uniform(0, 2 * np.pi, n_out)
    t_in = rng.uniform(0, 2 * np.pi, n_in)
    X = np.vstack([np.c_[np.cos(t_out), np.sin(t_out)],
                   factor * np.c_[np.cos(t_in), np.sin(t_in)]])
    X += rng.normal(0, noise, size=X.shape)
    y = np.r_[np.zeros(n_out, dtype=int), np.ones(n_in, dtype=int)]
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

## Implementation

In [ ]:
class DBSCAN:
    NOISE, UNVISITED = -1, -2

    def __init__(self, eps=0.5, min_samples=5):
        self.eps = eps
        self.min_samples = min_samples

    def fit(self, X):
        n = len(X)
        sq = (X ** 2).sum(1)
        D2 = sq[:, None] + sq[None, :] - 2 * X @ X.T
        neighbours = [np.flatnonzero(row <= self.eps ** 2) for row in D2]   # region queries (includes self)
        is_core = np.array([len(nb) >= self.min_samples for nb in neighbours])

        labels = np.full(n, self.UNVISITED)
        cluster = 0
        for p in range(n):
            if labels[p] != self.UNVISITED or not is_core[p]:
                continue
            labels[p] = cluster                     # new cluster seeded by a core point
            queue = deque(neighbours[p])
            while queue:
                q = queue.popleft()
                if labels[q] != self.UNVISITED:     # already in a cluster (a border point keeps its first cluster)
                    continue
                labels[q] = cluster
                if is_core[q]:                      # only core points keep expanding the cluster
                    queue.extend(neighbours[q])
            cluster += 1

        labels[labels == self.UNVISITED] = self.NOISE   # never reached from a core point
        self.labels_ = labels
        self.core_sample_indices_ = np.flatnonzero(is_core)
        return self

## Two moons with background noise

In [ ]:
Xm, _ = make_moons(n_samples=400, noise=0.06, random_state=0)
noise = np.random.RandomState(1).uniform([-1.5, -1], [2.5, 1.5], size=(40, 2))
X = np.vstack([Xm, noise])

db = DBSCAN(eps=0.2, min_samples=5).fit(X)
labels = db.labels_
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
print(f"clusters found: {n_clusters}, noise points: {np.sum(labels == -1)}")

core = np.zeros(len(X), dtype=bool); core[db.core_sample_indices_] = True
border = (labels != -1) & ~core
plt.figure(figsize=(8, 5.5))
plt.scatter(X[core, 0], X[core, 1], c=labels[core], cmap="viridis", s=30, label="core")
plt.scatter(X[border, 0], X[border, 1], c=labels[border], cmap="viridis", s=30, marker="^", edgecolors="k", label="border")
plt.scatter(X[labels == -1, 0], X[labels == -1, 1], c="red", marker="x", s=40, label="noise")
plt.legend(); plt.title("DBSCAN: core, border and noise points"); plt.show()

## Choosing `eps`: the k-distance plot
Sort every point's distance to its $k$-th nearest neighbour ($k$ = `min_samples` − 1, i.e. excluding the point itself).
The "knee" of the curve is a good $\varepsilon$ — beyond it distances jump because points are in sparse regions.

In [ ]:
def k_distance(X, k):
    D = np.sqrt(np.maximum((X ** 2).sum(1)[:, None] + (X ** 2).sum(1)[None, :] - 2 * X @ X.T, 0))
    return np.sort(np.sort(D, axis=1)[:, k])      # column 0 is the point itself

kd = k_distance(X, 4)
plt.plot(kd); plt.axhline(0.2, c="red", ls="--", label="eps = 0.2")
plt.xlabel("points sorted by 4-NN distance"); plt.ylabel("4-NN distance"); plt.legend()
plt.title("k-distance plot"); plt.show()

## Sensitivity to the parameters

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, (eps, ms) in zip(axes.ravel(), [(0.05, 5), (0.1, 5), (0.2, 5), (0.5, 5), (0.2, 2), (0.2, 20)]):
    lab = DBSCAN(eps, ms).fit(X).labels_
    k = len(set(lab)) - (1 if -1 in lab else 0)
    ax.scatter(X[lab != -1, 0], X[lab != -1, 1], c=lab[lab != -1], cmap="viridis", s=10)
    ax.scatter(X[lab == -1, 0], X[lab == -1, 1], c="red", marker="x", s=15)
    ax.set_title(f"eps={eps}, min_samples={ms}: {k} clusters, {np.sum(lab == -1)} noise")
plt.tight_layout(); plt.show()

## DBSCAN vs K-Means on non-convex shapes

In [ ]:
def kmeans_labels(X, k, n_iter=100, seed=0):
    rng = np.random.RandomState(seed)
    centers = X[rng.choice(len(X), k, replace=False)]
    for _ in range(n_iter):
        labels = np.argmin(((X[:, None] - centers[None]) ** 2).sum(-1), axis=1)
        centers = np.array([X[labels == j].mean(0) if np.any(labels == j) else centers[j] for j in range(k)])
    return labels

Xc, _ = make_circles(400, factor=0.4, noise=0.05, random_state=0)
Xmm, _ = make_moons(400, noise=0.05, random_state=0)
Xbb, _ = make_blobs(400, centers=3, cluster_std=0.6, random_state=2)

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for col, (Xd, k, eps, name) in enumerate([(Xc, 2, 0.15, "circles"), (Xmm, 2, 0.2, "moons"), (Xbb, 3, 0.6, "blobs")]):
    km = kmeans_labels(Xd, k)
    dbl = DBSCAN(eps, 5).fit(Xd).labels_
    axes[0, col].scatter(Xd[:, 0], Xd[:, 1], c=km, cmap="viridis", s=10); axes[0, col].set_title(f"K-Means — {name}")
    axes[1, col].scatter(Xd[:, 0], Xd[:, 1], c=dbl, cmap="viridis", s=10); axes[1, col].set_title(f"DBSCAN — {name}")
plt.tight_layout(); plt.show()

## Limitation: varying densities
A single global $\varepsilon$ cannot fit clusters of very different densities (HDBSCAN / OPTICS address this).

In [ ]:
Xvd, _ = make_blobs(450, centers=[[0, 0], [6, 0]], cluster_std=[0.3, 2.0], random_state=0)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, eps in zip(axes, [0.3, 1.0]):
    lab = DBSCAN(eps, 5).fit(Xvd).labels_
    ax.scatter(Xvd[lab != -1, 0], Xvd[lab != -1, 1], c=lab[lab != -1], cmap="viridis", s=10)
    ax.scatter(Xvd[lab == -1, 0], Xvd[lab == -1, 1], c="red", marker="x", s=12)
    ax.set_title(f"eps={eps}: {len(set(lab)) - (-1 in lab)} clusters, {np.sum(lab == -1)} noise")
plt.tight_layout(); plt.show()

## Summary
* DBSCAN finds clusters of any shape and labels outliers as noise; no need to specify $k$.
* Needs `eps` and `min_samples`; use the k-distance plot for `eps` and `min_samples` ≈ 2 × dimensions as a rule of thumb.
* Struggles with clusters of very different density and in high dimensions (distances become uniform).
* This naive version is $O(n^2)$; spatial indexes (KD-tree, ball tree) bring region queries down to roughly $O(n\log n)$.